# 生成式压缩器 · Round 0（QLoRA, Qwen3-0.6B）

**这是什么**：把「出生思维链 raw + 当时可见 ctx」压成更短思维链的**生成式模型**第一轮。

**训练数据（教师 v0.3）**：目标由**装上了尺子的机械教师**生成 —— 只保留「看稿子就能判」的 7 条轴：
`M1` 压缩 ≤0.60 · `M3` 闭合判读 · `M4` 可执行验收（逐字命令+两分支） · `M5` 接地精度 · `M6` 无装置话术 · `M7` 决策不变 · `M8` 落点唯一。
（`E1/E2` 真机、`R1/R2` 台账、`M2` 产线对照——凡需要「另外跑」的轴**全部剔除**，按用户口径。）

**训练行**：937 条、623 仓库，**全部 7 轴通过**（不合规的 106 条另存 `train-nonconforming.jsonl.gz`，不进训练）。
**dev**：139 条 / 86 仓库（与 train 仓库不相交），其中 124 条同样 7 轴全过。

**诚实边界**：① 目标稿是机器生成，**不是**已审校金标（人工审核 = 0）；② 教师用的是同一把尺子 ⇒ 之后量学生是**同源测量**（测学没学到形状），**不构成对尺子本身的验证**；③ `E1/E2` 未测，需回喂目标模型（DeepSeek-V4.1-Flash 端点）。


In [ ]:
import sys, torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())
assert torch.cuda.is_available(), 'GPU 必开（Notebook 设置 → Accelerator → GPU T4 x2）'


In [ ]:
import subprocess
pkgs = ['transformers>=4.51,<5', 'peft>=0.15,<1', 'accelerate>=1.6,<2', 'bitsandbytes>=0.45,<1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *pkgs], check=True)
print('deps ok')


In [ ]:
from pathlib import Path

# 数据集挂载点（把我们上传的 kaggle-upload 文件夹作为 dataset 挂上；名字可改）
CANDIDATES = sorted(Path('/kaggle/input').glob('*/train.jsonl.gz'))
assert CANDIDATES, '没找到挂载的数据集：确认已 Add Input → 本仓库的 kaggle-upload 数据集'
DATA = CANDIDATES[0].parent
TRAINER = DATA / 'train_gen_kaggle.py'
assert TRAINER.exists(), f'数据集里缺 train_gen_kaggle.py：{DATA}'
print('data dir :', DATA)
for f in sorted(DATA.iterdir()): print('   ', f.name, f.stat().st_size, 'bytes')


In [ ]:
# 训练（T4×2，1,039 条 × 2 epoch，估 ~40-70 分钟；日志如实保留）
import subprocess, sys
OUT = Path('/kaggle/working/run1')
cmd = [sys.executable, str(TRAINER),
       '--train', str(DATA/'train.jsonl.gz'), '--dev', str(DATA/'dev.jsonl.gz'),
       '--out', str(OUT), '--model-id', 'Qwen/Qwen3-0.6B',
       '--epochs', '2', '--lr', '1e-4', '--max-len', '5120', '--batch', '1', '--grad-accum', '16',
       '--eval-samples', '12']
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 读数：关键锚点留存（未压缩块 → 生成的压缩块），外加两条样本
import json
meta = json.loads((OUT/'run-meta.json').read_text())
print(json.dumps({k: meta[k] for k in ['trainRows','trainExamplesUsed','skippedOverMaxLen','finalLossAvg50','trainingSeconds','devMetrics']}, ensure_ascii=False, indent=2))
preds = [json.loads(l) for l in (OUT/'dev-predictions.jsonl').read_text().split('\n') if l.strip()]
for r in preds[:2]:
    print('─'*90); print(r['unitId'], '| repo', r['repo'], '| pred', r['predChars'], 'chars | critCov', r['vsRaw']['criticalCoverage'], '| ratio', r['vsRaw']['ratio'])
    print(r['prediction'][:700])


In [ ]:
# 打包产物（adapter + 读数 + 预测）供下载
import shutil
archive = shutil.make_archive('/kaggle/working/cfb-gen-compressor-run1', 'zip', root_dir=OUT)
print('download:', archive, Path(archive).stat().st_size, 'bytes')
print('目录内容:', sorted(p.name for p in OUT.iterdir()))
